# 沪深300 PE+PB估值策略 分析笔记本
每月运行一次，查看当前信号和历史表现。

**v2 改动**：双因子（PE+PB）合成估值分位 / 线性仓位 / 趋势过滤（MA250）/ Walk-forward验证

In [ ]:
# ── 0. 依赖检查 ───────────────────────────────────────────
import subprocess, sys
pkgs = ['akshare', 'pandas', 'numpy', 'matplotlib']
for p in pkgs:
    try:
        __import__(p)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p])
print('依赖就绪')

In [ ]:
# ── 1. 拉取最新数据 ────────────────────────────────────────
from fetch_data import fetch_price, fetch_pe, fetch_pb, load_merged

fetch_price()
fetch_pe()
fetch_pb()
df = load_merged()
print(f'数据范围：{df["date"].iloc[0].date()} → {df["date"].iloc[-1].date()}，共 {len(df)} 条')

In [ ]:
# ── 2. 当前信号 ────────────────────────────────────────────
from strategy import current_signal

sig = current_signal()
print('\n=== 当前信号 ===')
print(f"日期            : {sig['date']}")
print(f"沪深300收盘     : {sig['close']}")
print(f"PE (TTM)        : {sig['pe']}  （历史百分位 {sig['pe_pct']}%）")
print(f"PB              : {sig['pb']}  （历史百分位 {sig['pb_pct']}%）")
print(f"合成估值分位    : {sig['composite_pct']}%")
print(f"趋势（>MA250）  : {'✓ 多头' if sig['trend_up'] else '✗ 空头（趋势过滤触发）'}")
print(f"估值建议仓位    : {sig['raw_position']}%")
print(f"▶ 最终建议仓位  : {sig['position']}%")

In [ ]:
# ── 3. 回测 ────────────────────────────────────────────────
from backtest import run_backtest, calc_metrics

bt = run_backtest()
metrics = calc_metrics(bt)

print('\n=== 回测绩效 ===')
for k, v in metrics.items():
    print(f'{k}: {v}')

In [ ]:
# ── 4. 净值曲线 ────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)
fig.suptitle('沪深300 PE+PB估值策略 (v2)', fontsize=14, fontweight='bold')

# 净值曲线
ax1 = axes[0]
ax1.plot(bt['date'], bt['strategy_nav'],  label='策略', color='steelblue', linewidth=1.5)
ax1.plot(bt['date'], bt['benchmark_nav'], label='基准（满仓）', color='gray', linewidth=1, alpha=0.7)
ax1.set_ylabel('净值（元）')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 仓位
ax2 = axes[1]
ax2.fill_between(bt['date'], bt['position'], alpha=0.6, color='steelblue', label='实际仓位')
ax2.plot(bt['date'], bt['raw_position'], color='orange', linewidth=0.8, alpha=0.7, label='估值仓位（过滤前）')
ax2.set_ylabel('仓位')
ax2.set_ylim(0, 1.1)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(y*100)}%'))
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# 合成估值分位
ax3 = axes[2]
ax3.plot(bt['date'], bt['composite_pct'], color='darkorange', linewidth=1, label='合成估值分位')
ax3.plot(bt['date'], bt['pe_pct'],        color='steelblue',  linewidth=0.8, alpha=0.6, label='PE分位')
ax3.plot(bt['date'], bt['pb_pct'],        color='green',      linewidth=0.8, alpha=0.6, label='PB分位')
ax3.axhline(0.2, color='green', linestyle='--', alpha=0.5)
ax3.axhline(0.8, color='red',   linestyle='--', alpha=0.5)
ax3.set_ylabel('历史百分位')
ax3.set_ylim(0, 1)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(y*100)}%'))
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# 趋势信号
ax4 = axes[3]
ax4.plot(bt['date'], bt['close'], color='gray', linewidth=1, label='收盘价')
ax4.plot(bt['date'], bt['ma250'], color='red',  linewidth=1.2, label='MA250')
ax4.fill_between(bt['date'], bt['close'], bt['ma250'],
                  where=bt['trend_up'], alpha=0.15, color='green', label='多头区间')
ax4.set_ylabel('价格')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('reports/strategy_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存至 reports/strategy_chart.png')

In [ ]:
# ── 5. 回撤分析 ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

s_dd = (bt['strategy_nav']  / bt['strategy_nav'].cummax()  - 1) * 100
b_dd = (bt['benchmark_nav'] / bt['benchmark_nav'].cummax() - 1) * 100

ax.fill_between(bt['date'], s_dd, label='策略回撤', color='steelblue', alpha=0.6)
ax.fill_between(bt['date'], b_dd, label='基准回撤', color='gray',      alpha=0.4)
ax.set_ylabel('回撤（%）')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.title('回撤对比')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. （可选）参数优化 + Walk-forward 验证 ────────────────
# 耗时较长，按需运行
# from optimize import run_optimization, run_walk_forward
# results = run_optimization()
# display(results.head(10))
# wf = run_walk_forward()
# display(wf)

In [ ]:
# ── 7. 保存报告 ────────────────────────────────────────────
from report import save_report
save_report(bt, sig, metrics)
print('完成')